In [1]:
import pandas as pd
import sklearn
print(pd.__version__)
print(sklearn.__version__)

2.3.3
1.6.1


In [2]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent      # notebooks/ -> project root
sys.path.insert(0, str(ROOT))

In [3]:
columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
    "label", "difficulty"
]

print(len(columns))


df_train = pd.read_csv("../data/raw/KDDTrain+.txt", header=None, names=columns)
df_test = pd.read_csv("../data/raw/KDDTest+.txt", header=None, names=columns)
print(f" Shape of training data: {df_train.shape}")
print(f" Shape of test data: {df_test.shape}")

43


 Shape of training data: (125973, 43)
 Shape of test data: (22544, 43)


## Isolation Forest baseline

Isolation Forest flags points that are easy to separate from the rest of the data. It is trained here two ways:

1. **Mixed training** — on all training rows (normal + attacks)
2. **Normal-only training** — only on normal traffic, so attacks look unusual

The threshold is the 95th percentile of scores on normal training traffic, the same rule used for Mahalanobis and the autoencoder.

In [4]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

x_train = df_train.drop(["label", "difficulty"], axis=1)
x_test = df_test.drop(["label", "difficulty"], axis=1)
y_train_binary = (df_train["label"] != "normal").astype(int)
y_test_binary = (df_test["label"] != "normal").astype(int)

# one-hot encode protocol_type, service, flag; test gets the same columns as train
x_train_encoded = pd.get_dummies(x_train)
x_test_encoded = pd.get_dummies(x_test).reindex(columns=x_train_encoded.columns, fill_value=0)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train_encoded)
x_test_scaled = scaler.transform(x_test_encoded)

x_normal = x_train_scaled[(y_train_binary == 0).to_numpy()]
print(x_train_scaled.shape, x_test_scaled.shape, x_normal.shape)

(125973, 122) (22544, 122) (67343, 122)


In [5]:
def evaluate(name, iso):
    # score_samples is higher for normal points, so flip it: HIGHER = more likely an attack
    normal_scores = -iso.score_samples(x_normal)
    test_scores = -iso.score_samples(x_test_scaled)

    threshold = np.percentile(normal_scores, 95)  # 5% of normal training traffic gets flagged
    y_pred = (test_scores > threshold).astype(int)

    print(f"--- {name}")
    print("threshold:", threshold)
    print(confusion_matrix(y_test_binary, y_pred))
    print("Precision:", precision_score(y_test_binary, y_pred))
    print("Recall:", recall_score(y_test_binary, y_pred))
    print("F1 Score:", f1_score(y_test_binary, y_pred))
    print("ROC AUC:", roc_auc_score(y_test_binary, test_scores))
    return threshold

In [6]:
# 1. Mixed training: attacks are in the training data, so the forest learns them as "normal"
iso_mixed = IsolationForest(random_state=42).fit(x_train_scaled)
evaluate("Mixed training", iso_mixed)

# sklearn's built-in predict() uses its own cutoff, which flags almost nothing here
y_pred_builtin = (iso_mixed.predict(x_test_scaled) == -1).astype(int)
print("Recall with built-in predict():", recall_score(y_test_binary, y_pred_builtin))

--- Mixed training
threshold: 0.42742822531146457
[[9186  525]
 [9890 2943]]
Precision: 0.8486159169550173
Recall: 0.2293306319644666
F1 Score: 0.36108214219986506
ROC AUC: 0.8652866405309982
Recall with built-in predict(): 7.792410192472531e-05


In [7]:
# 2. Normal-only training
iso = IsolationForest(random_state=42).fit(x_normal)
threshold_iso = evaluate("Normal-only training", iso)

--- Normal-only training
threshold: 0.4835411207880887
[[9068  643]
 [4150 8683]]
Precision: 0.9310529701908643
Recall: 0.6766149770123899
F1 Score: 0.7836996254343608
ROC AUC: 0.9382087710024252


In [8]:
# 3. Normal-only, tuned: more trees and bigger samples per tree
iso_tuned = IsolationForest(n_estimators=300, max_samples=1024, random_state=42).fit(x_normal)
threshold_iso_tuned = evaluate("Normal-only, 300 trees, max_samples=1024", iso_tuned)

--- Normal-only, 300 trees, max_samples=1024
threshold: 0.45641773156554916
[[9046  665]
 [3788 9045]]
Precision: 0.931513903192585
Recall: 0.7048235019091404
F1 Score: 0.8024663975513463
ROC AUC: 0.9479012060726749


### Results on KDDTest+

| Training | Precision | Recall | F1 | ROC AUC |
|---|---|---|---|---|
| Mixed (normal + attacks) | 0.849 | 0.229 | 0.361 | 0.865 |
| Normal-only | 0.931 | 0.677 | 0.784 | 0.938 |
| Normal-only, 300 trees, max_samples=1024 | 0.932 | 0.705 | 0.802 | 0.948 |

- Training on **normal traffic only** triples recall. With attacks in the training data, the forest learns them as part of "normal".
- sklearn's built-in `predict()` on the mixed model flags almost nothing (recall ≈ 0.0001), so a threshold set from normal traffic is needed.
- Isolation Forest still has lower recall than Mahalanobis (0.847) and the autoencoder (0.842).

In [9]:
from src.detection.preprocess import load_preprocessor, preprocess
from src.detection.detectors import load_all

scaler, cols = load_preprocessor("../artifacts")
dets = load_all("../artifacts")

X = preprocess(df_test, scaler, cols)
out = dets["isolation_forest"].predict(X)
print(out.keys())               # check the field names before using them
scores = out["score"]           # higher = more anomalous
y = (df_test["label"] != "normal").astype(int)
print(scores.shape, y.shape)    

dict_keys(['score', 'flagged', 'ratio'])
(22544,) (22544,)


In [10]:
from sklearn.model_selection import train_test_split
scores_A, scores_B, y_A, y_B = train_test_split(
    scores, y, test_size=0.5, stratify=y, random_state=42
)
print(scores_A.shape, scores_B.shape)
print(y_A.mean(), y_B.mean())

(11272,) (11272,)
0.5692867281760113 0.5691980127750177


In [11]:
A = scores_A.reshape(-1,1)
B = scores_B.reshape(-1,1)


In [12]:
from sklearn.linear_model import LogisticRegression

cal = LogisticRegression()   # blank model, nothing learned yet
cal.fit(A, y_A)    
print(cal.coef_, cal.intercept_)# learns w and b from half A

[[20.90580336]] [-8.66988761]


/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights


In [13]:
import numpy as np
print(np.isfinite(A).all(), np.isfinite(y_A).all())   # both should be True
cal2 = LogisticRegression(solver="liblinear")
cal2.fit(A, y_A)
print(cal2.coef_, cal2.intercept_)

True True
[[20.49867419]] [-8.49790438]


In [14]:
from sklearn.metrics import brier_score_loss
p_B = cal.predict_proba(B)[:, 1]
print(brier_score_loss(y_B, p_B))

0.10464067346266344


In [15]:
from sklearn.calibration import calibration_curve

frac_attack, mean_pred = calibration_curve(y_B, p_B, n_bins=10)
table = pd.DataFrame({
    "predicted": mean_pred,
    "actual" :frac_attack,
})
print(table.round(3))

   predicted  actual
0      0.137   0.030
1      0.242   0.439
2      0.345   0.404
3      0.458   0.596
4      0.555   0.672
5      0.644   0.747
6      0.743   0.846
7      0.863   0.690
8      0.962   0.983


In [16]:
counts = np.histogram(p_B, bins=np.linspace(0, 1, 11))[0]
table["n"] = counts[counts > 0]

In [17]:
from sklearn.isotonic import IsotonicRegression
iso_cal = IsotonicRegression(out_of_bounds="clip")
iso_cal.fit(scores_A, y_A)
p_B_iso = iso_cal.predict(scores_B)

In [18]:
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss

# --- 1. logistic calibrator (what you already have) ---
# cal and p_B already exist from your earlier cells

# --- 2. isotonic calibrator ---
iso_cal = IsotonicRegression(out_of_bounds="clip")
iso_cal.fit(scores_A, y_A)            # flat 1D scores, not the reshaped A
p_B_iso = iso_cal.predict(scores_B)   # returns probabilities directly

# --- 3. reliability table with counts ---
def reliability(y_true, p, n_bins=10):
    y_true = np.asarray(y_true)
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.digitize(p, edges[1:-1])          # bin number 0..n_bins-1 for each row
    rows = []
    for b in range(n_bins):
        m = idx == b                            # True for rows in this bin
        if m.sum() == 0:
            continue
        rows.append((b, p[m].mean(), y_true[m].mean(), int(m.sum())))
    return pd.DataFrame(rows, columns=["bin", "predicted", "actual", "n"])

# --- 4. compare ---
print("Brier logistic:", brier_score_loss(y_B, p_B))
print("Brier isotonic:", brier_score_loss(y_B, p_B_iso))
print("\nLOGISTIC")
print(reliability(y_B, p_B).round(3))
print("\nISOTONIC")
print(reliability(y_B, p_B_iso).round(3))

Brier logistic: 0.10464067346266344
Brier isotonic: 0.08999750071173593

LOGISTIC
   bin  predicted  actual     n
0    1      0.137   0.030  3198
1    2      0.242   0.439   684
2    3      0.345   0.404   834
3    4      0.458   0.596   681
4    5      0.555   0.672   293
5    6      0.644   0.747   545
6    7      0.743   0.846   332
7    8      0.863   0.690   802
8    9      0.962   0.983  3903

ISOTONIC
   bin  predicted  actual     n
0    0      0.000   0.000  2987
1    2      0.278   0.407    27
2    3      0.390   0.439    41
3    4      0.429   0.428  1885
4    5      0.542   0.500    14
5    6      0.680   0.678  2040
6    8      0.855   0.939   147
7    9      0.977   0.981  4131


In [19]:
def calibrate(name):
    out = dets[name].predict(X)
    s = np.asarray(out["score"])
    sA, sB, yA, yB = train_test_split(s, y, test_size=0.5,
                                      stratify=y, random_state=42)
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0.01, y_max=0.99)
    iso.fit(sA, yA)
    p = iso.predict(sB)
    print(name, "Brier:", round(brier_score_loss(yB, p), 4))
    print(reliability(yB, p).round(3))
    return iso, p, yB

iso_if,  p_if,  y_B = calibrate("isolation_forest")

isolation_forest Brier: 0.09
   bin  predicted  actual     n
0    0      0.010   0.000  2987
1    2      0.278   0.407    27
2    3      0.390   0.439    41
3    4      0.429   0.428  1885
4    5      0.542   0.500    14
5    6      0.680   0.678  2040
6    8      0.855   0.939   147
7    9      0.975   0.981  4131
